# AI-Publisher Docker Image Builder v2

Bu notebook, AI-Publisher platformu icin Docker imajlarini Google Colab'da Kaniko ile build eder.
Imajlar Google Drive'a `.tar.gz` olarak kaydedilir, yerel makinede `docker compose up` ile yuklenir.

### Degisiklikler (v2):
- **Video codec fix**: cogvideox, wan, ltx, hunyuan, svd, animatediff, wan25 -> MJPG -> libx264 + yuv420p
- **Colab runtime kalkti**: Bu notebook sadece build icin, runtime sunucu Docker container'larda
- **Force-Rebuild**: Video modelleri zorunlu rebuild edilir (codec fix icin)
- **Incremental**: Ses/efekt modelleri Drive'da varsa atlanir

---
Her hucreyi sirayla calistirin.

## Hucre 1: Runtime Kontrolu

Sistem durumunu kontrol et. Build tamamen CPU calistigi icin GPU runtime gerekmez.
Runtime > Change runtime type > None secin.

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Runtime Kontrolu { display-mode: "form" }
import subprocess
import os

def check_runtime():
    print("=" * 60)
    print("AI-Publisher Docker Build - Runtime Kontrolu")
    print("=" * 60)
    
    # GPU kontrolu
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=10
        , env=_clean_env)
        if result.returncode == 0:
            gpu_info = result.stdout.strip().split(", ")
            print(f"GPU: {gpu_info[0]}")
            print(f"   Toplam VRAM: {gpu_info[1]} MB")
            print(f"   Bos VRAM: {gpu_info[2]} MB")
        else:
            print("GPU bulunamadi. Build icin GPU gerekli degil, devam.")
    except FileNotFoundError:
        print("nvidia-smi bulunamadi. Build icin GPU gerekli degil, devam.")
    
    # Disk alani
    disk = subprocess.run(["df", "-h", "/content"], capture_output=True, text=True, env=_clean_env)
    if disk.returncode == 0:
        lines = disk.stdout.strip().split("\n")
        if len(lines) > 1:
            parts = lines[1].split()
            print(f"Disk: {parts[2]} kullanildi / {parts[1]} toplam")
    
    # RAM
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal"):
                total_kb = int(line.split()[1])
                print(f"RAM: {total_kb // 1024} MB")
                break
    
    print("\nRuntime hazir. Sonraki hucreye gecin.")
    return True

check_runtime()

## Hucre 2: Drive Mount ve Bagimliliklar

Google Drive'i bagla, Kaniko + registry + pigz kur.

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Drive Mount ve Bagimliliklar { display-mode: "form" }
import subprocess
import sys
import os
import time

print("Bagimliliklar kuruluyor...")

# pip kontrol
subprocess.run("rm -f /var/lib/dpkg/lock-frontend /var/lib/apt/lists/lock /var/cache/apt/archives/lock 2>/dev/null || true", shell=True, env=_clean_env)
subprocess.run("dpkg --configure -a 2>/dev/null || true", shell=True, env=_clean_env)
r = subprocess.run([sys.executable, "-m", "pip", "--version"], capture_output=True, text=True, env=_clean_env)
if r.returncode != 0:
    print("  pip bulunamadi, apt-get ile kuruluyor...")
    subprocess.run("apt-get update -q && apt-get install -y -q python3-pip", shell=True, check=True, timeout=120, env=_clean_env)

# Google Drive mount
try:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")
        print("Google Drive baglandi.")
    else:
        print("Google Drive zaten bagli.")
except Exception as e:
    print(f"Drive mount hatasi: {e}")

print("\nKaniko + Registry + pigz kurulumu...")

# Kaniko kurulumu
kaniko_checks = [
    {"path": "/kaniko/executor", "label": "Kaniko"},
    {"path": "/usr/local/bin/registry", "label": "Registry"},
    {"path": "/usr/local/bin/pigz", "label": "pigz"},
]
all_ok = all(os.path.exists(c["path"]) for c in kaniko_checks)

if not all_ok:
    subprocess.run(["mkdir", "-p", "/kaniko"], check=True, env=_clean_env)
    subprocess.run(["apt-get", "update", "-qq"], check=True, timeout=120, env=_clean_env)
    
    # 1. Kaniko - GCR imajindan binary cikar (Docker gerekmez)
    print("  Kaniko GCR imajindan cikariliyor...")
    import requests, tarfile, io, stat
    
    kaniko_dest = '/kaniko/executor'
    if not os.path.exists(kaniko_dest):
        idx_resp = requests.get(
            'https://gcr.io/v2/kaniko-project/executor/manifests/latest',
            headers={'Accept': 'application/vnd.oci.image.index.v1+json'},
            timeout=30
        )
        idx_resp.raise_for_status()
        idx_data = idx_resp.json()
        
        amd64_digest = None
        for m in idx_data['manifests']:
            if m['platform']['architecture'] == 'amd64':
                amd64_digest = m['digest']
                break
        
        if not amd64_digest:
            raise RuntimeError('AMD64 manifest not found')
        
        manifest_resp = requests.get(
            f'https://gcr.io/v2/kaniko-project/executor/manifests/{amd64_digest}',
            headers={'Accept': 'application/vnd.oci.image.manifest.v1+json'},
            timeout=30
        )
        manifest_resp.raise_for_status()
        manifest_data = manifest_resp.json()
        
        found = False
        for i, layer in enumerate(manifest_data['layers']):
            digest = layer['digest']
            url = f'https://gcr.io/v2/kaniko-project/executor/blobs/{digest}'
            print(f'    Layer {i+1}/{len(manifest_data["layers"])} indiriliyor...')
            
            layer_resp = requests.get(url, timeout=120)
            layer_resp.raise_for_status()
            
            try:
                with tarfile.open(fileobj=io.BytesIO(layer_resp.content), mode='r:gz') as tar:
                    for member in tar.getmembers():
                        if member.name == 'kaniko/executor':
                            print(f'    kaniko/executor bulundu (layer {i+1})')
                            f = tar.extractfile(member)
                            if f:
                                with open(kaniko_dest, 'wb') as out:
                                    out.write(f.read())
                                st = os.stat(kaniko_dest)
                                os.chmod(kaniko_dest, st.st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
                                found = True
                                break
            except tarfile.ReadError:
                continue
            if found:
                break
        
        if not found:
            raise RuntimeError('kaniko/executor not found in GCR layers')
        
        if os.path.exists('/usr/local/bin/kaniko'):
            os.remove('/usr/local/bin/kaniko')
        os.symlink(kaniko_dest, '/usr/local/bin/kaniko')
        size_mb = os.path.getsize(kaniko_dest) / (1024 * 1024)
        print(f'    Kaniko kuruldu ({size_mb:.1f} MB)')
    else:
        print('    Kaniko zaten mevcut.')
    
    # 2. Registry - dist distribution binary
    print("  Registry indiriliyor...")
    if not os.path.exists('/usr/local/bin/registry'):
        import urllib.request
        url = 'https://github.com/distribution/distribution/releases/download/v2.8.3/registry_2.8.3_linux_amd64.tar.gz'
        try:
            urllib.request.urlretrieve(url, '/tmp/registry.tar.gz')
            import tarfile as tfgz
            with tfgz.open('/tmp/registry.tar.gz', 'r:gz') as tar:
                for member in tar.getmembers():
                    if member.name == 'registry':
                        f = tar.extractfile(member)
                        if f:
                            with open('/usr/local/bin/registry', 'wb') as out:
                                out.write(f.read())
                            os.chmod('/usr/local/bin/registry', 0o755)
                            print('    Registry kuruldu.')
                            break
            os.remove('/tmp/registry.tar.gz')
        except Exception as e:
            print(f'    WARN: Registry indirilemedi: {e}')
    else:
        print('    Registry zaten mevcut.')
    
    # 3. pigz
    print("  pigz kuruluyor...")
    subprocess.run(["apt-get", "install", "-y", "-qq", "pigz"], timeout=60, env=_clean_env)
    
    # 4. Registry config
    os.makedirs('/etc/docker/registry', exist_ok=True)
    os.makedirs('/var/lib/registry', exist_ok=True)
    with open('/etc/docker/registry/config.yml', 'w') as f:
        f.write('version: 0.1\n')
        f.write('log:\n')
        f.write('  level: error\n')
        f.write('storage:\n')
        f.write('  filesystem:\n')
        f.write('    rootdirectory: /var/lib/registry\n')
        f.write('http:\n')
        f.write('  addr: :5000\n')
    print("  Kaniko + Registry + pigz hazir.")
    print("Kaniko, Registry ve pigz basariyla kuruldu.")
else:
    print("Kaniko, Registry ve pigz zaten mevcut.")

print("\nDrive mount ve bagimliliklar tamam. Sonraki hucreye gecin.")

## Hucre 3: Repo Klonlama

GitHub'dan en son kodu cek. Codec fix'in dahil oldugundan emin olmak icin `main` branch'ini kullan.

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Repo Klonlama { display-mode: "form" }
import subprocess
import os

REPO_URL = "https://github.com/Arda-Avci/AI-Publisher.git"
WORK_DIR = "/content/AI-Publisher"

print("Repo klonlaniyor...")

# GitHub PAT - sirasiyla dene: Colab secrets -> env var
GITHUB_PAT = ""

# 1) Colab secrets
try:
    from google.colab import userdata
    for name in ["GITHUB_PAT", "GITHUB_TOKEN", "GH_TOKEN"]:
        try:
            val = userdata.get(name)
            if val:
                GITHUB_PAT = val
                print(f"Colab secrets'dan '{name}' alindi.")
                break
        except Exception as e:
            print(f"  userdata.get('{name}') -> {e}")
except ImportError:
    print("google.colab.userdata mevcut degil (Colab disi ortam).")

# 2) Ortam degiskeni fallback
if not GITHUB_PAT:
    for name in ["GITHUB_PAT", "GITHUB_TOKEN", "GH_TOKEN"]:
        val = os.environ.get(name)
        if val:
            GITHUB_PAT = val
            print(f"Ortam degiskeni '{name}' alindi.")
            break

if not GITHUB_PAT:
    print("\n" + "="*60)
    print("HICBIR GITHUB TOKEN BULUNAMADI!")
    print("="*60)
    print("Colab Secrets'a ekleyin (SOLDAN: Anahtar simgesi -> Add new secret):")
    print("  Name: GITHUB_PAT")
    print("  Value: <github_pat>")
    print("  Notebook access: ON")
    print("\nAlternatif: Asagidaki forma token'i yapistirip calistirabilirsiniz.")
    try:
        from IPython.display import display
        import ipywidgets as widgets
        pat_input = widgets.Password(description="GITHUB_PAT:", style={"description_width": "initial"})
        display(pat_input)
        print("Token'i yazip hucreyi yeniden calistirin.")
    except ImportError:
        pass
    raise ValueError("GITHUB_PAT bulunamadi! Colab Secrets'a ekleyin.")

if os.path.exists(WORK_DIR):
    print("Repo zaten klonlanmis, guncelleniyor...")
    os.chdir(WORK_DIR)
    subprocess.run(["git", "fetch", "origin"], check=True, timeout=30, env=_clean_env)
    subprocess.run(["git", "reset", "--hard", "origin/main"], check=True, timeout=30, env=_clean_env)
    print("Repo guncellendi.")
else:
    auth_url = REPO_URL.replace("https://", f"https://{GITHUB_PAT}@")
    subprocess.run(["git", "clone", auth_url, WORK_DIR], check=True, timeout=60, env=_clean_env)
    os.chdir(WORK_DIR)
    print("Repo klonlandi.")

# Build script'inin var oldugunu dogrula
os.chdir(WORK_DIR + "/colab_docker")
assert os.path.exists("build_all.sh"), "build_all.sh bulunamadi!"
subprocess.run(["chmod", "+x", "build_all.sh"], check=True, env=_clean_env)
print(f"build_all.sh hazir. Commit: {subprocess.run(['git', 'log', '-1', '--oneline'], capture_output=True, text=True, env=_clean_env).stdout.strip()}")

## Hucre 4: Force-Rebuild Base + Video Modelleri

Video modelleri (codec fix iceren) zorunlu rebuild edilir.
Sirasiyla: base (altyapi), cogvideox, wan, ltx, hunyuan, svd, animatediff, wan25

> **Tahmini sure**: Base ~5dk, her video modeli ~15-30dk, toplam ~2 saat
> Colab baglantisi koparsa, bu hucreyi yeniden calistir (var olan imajlar atlanir)

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Force-Rebuild: Base + Video Modelleri { display-mode: "form" }
import subprocess
import os
import time

os.chdir("/content/AI-Publisher/colab_docker")

VIDEO_MODELS = [
    "base",
    "cogvideox",
    "wan",
    "ltx",
    "hunyuan",
    "svd",
    "animatediff",
    "wan25",
]

print("Force-Rebuild basliyor: Video modelleri (codec fix ile)")
print(f"Modeller: {', '.join(VIDEO_MODELS)}")
print("=" * 60)

total_start = time.time()

# force_rebuild.sh ile calistir (Drive arsivlerini siler)
result = subprocess.run(
    ["bash", "force_rebuild.sh"] + VIDEO_MODELS,
    capture_output=False,
    text=True,
    timeout=72000  # 20 saat timeout
, env=_clean_env)

total_duration = (time.time() - total_start) / 60
print(f"\nVideo model build tamamlandi. Toplam sure: {total_duration:.1f} dk")
print(f"Cikis kodu: {result.returncode}")

## Hucre 5: Incremental Build - Ses/Efekt Modelleri

Ses ve efekt modelleri incremental build edilir. Drive'da varsa atlanir.
**Bagimsizdir** - video build'u beklemez, ayri oturumda calistirilabilir.

Modeller: xtts, audioldm2, wav2lip, musetalk, whisper, stablediffusion, kokorotts, f5tts, lora-trainer

In [ ]:
import os
_clean_env = os.environ.copy()
_clean_env.pop('LD_LIBRARY_PATH', None) # GLIBC hatasini onlemek icin
#@title Incremental Build: Ses/Efekt Modelleri { display-mode: "form" }
import subprocess
import os
import time

os.chdir("/content/AI-Publisher/colab_docker")

print("Incremental build basliyor: Ses/Efekt modelleri")
print("(Drive'da var olan imajlar atlanacak)")
print("=" * 60)

total_start = time.time()

result = subprocess.run(
    ["bash", "build_all.sh"],
    capture_output=False,
    text=True,
    timeout=43200  # 12 saat
, env=_clean_env)

total_duration = (time.time() - total_start) / 60
print(f"\nSes/Efekt model build tamamlandi. Toplam sure: {total_duration:.1f} dk")
print(f"Cikis kodu: {result.returncode}")

## Hucre 6: Imaj Dogrulama

Drive'daki imajlari kontrol et: boyut, tarih, sayi.
Video modellerinde codec fix'i dogrula (opsiyonel).

In [ ]:
#@title Imaj Dogrulama { display-mode: "form" }
import subprocess
import os
import json

DRIVE_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/images"
MIN_SIZE = 100 * 1024 * 1024  # 100 MB

print("Drive'daki imajlar kontrol ediliyor...")
print("=" * 60)

if not os.path.exists(DRIVE_DIR):
    print(f"HATA: {DRIVE_DIR} bulunamadi! Drive mount edilmemis olabilir.")
else:
    import glob
    files = sorted(glob.glob(os.path.join(DRIVE_DIR, "*.tar.gz")))
    print(f"Toplam {len(files)} imaj dosyasi bulundu.\n")
    
    video_models = ["base", "cogvideox", "wan", "ltx", "hunyuan", "svd", "animatediff", "wan25"]
    audio_models = ["xtts", "audioldm2", "wav2lip", "musetalk", "whisper", "stablediffusion", "kokorotts", "f5tts", "lora-trainer"]
    
    # Model bazli durum
    all_models = video_models + audio_models
    drive_map = {os.path.basename(f).replace(".tar.gz", ""): f for f in files}
    
    for model in all_models:
        if model in drive_map:
            fpath = drive_map[model]
            size = os.path.getsize(fpath)
            size_mb = size / (1024 * 1024)
            mtime = os.path.getmtime(fpath)
            import datetime
            date_str = datetime.datetime.fromtimestamp(mtime).strftime("%Y-%m-%d %H:%M")
            status = "OK" if size >= MIN_SIZE else "KUCUK"
            print(f"  [{status:6s}] {model:20s} {size_mb:8.1f} MB  {date_str}")
        else:
            print(f"  [  EKSIK] {model:20s} Drive'da bulunamadi!")
    
    print()
    missing = [m for m in all_models if m not in drive_map]
    small = [m for m in all_models if m in drive_map and os.path.getsize(drive_map[m]) < MIN_SIZE]
    
    if not missing and not small:
        print("Tum imajlar basariyla build edilmis ve Drive'a kaydedilmis.")
    else:
        if missing:
            print(f"EKSIK: {', '.join(missing)}")
        if small:
            print(f"KUCUK/BOZUK: {', '.join(small)}")

print("\nDogrulama tamamlandi.")

## Hucre 7: GHCR'a Push Et

Drive'daki `.tar.gz` imajlari GHCR'a (GitHub Container Registry) yuklenir.

> **Gereken Colab Secret:** `GITHUB_PAT` (veya `GITHUB_TOKEN`, `GH_TOKEN`) — `write:packages` izni olmali
>
> Bu adim basarisiz olursa, `colab_docker/colab_ghcr_upload.ipynb` ile sonradan tekrar denenebilir.

In [ ]:
#@title GHCR Push { display-mode: "form" }
import time, shutil, pathlib, subprocess, sys, json, requests, base64
from pathlib import Path
from google.colab import userdata

DRIVE_IMAGE_DIR = "/content/drive/MyDrive/Colab Notebooks/docker/images"
GHCR_NAMESPACE = "ghcr.io/arda-avci"
IMAGE_PREFIX = "ai-publisher-"
TEMP_DIR = Path("/content/tmp_images")
TEMP_DIR.mkdir(parents=True, exist_ok=True)

def _ts():
    return time.strftime("%H:%M:%S")

def stream_cmd(cmd, timeout=None, step=""):
    print(f"  [{_ts()}] {step}")
    sys.stdout.flush()
    proc = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1, text=True)
    lines = []
    for line in proc.stdout:
        line = line.strip()
        if line:
            lines.append(line)
            print(f"    {line[:120]}")
            sys.stdout.flush()
    proc.wait()
    return proc.returncode, "\n".join(lines)

def run_cap(cmd, timeout=None):
    kwargs = dict(shell=True, text=True, capture_output=True)
    if timeout: kwargs["timeout"] = timeout
    try: r = subprocess.run(cmd, **kwargs)
    except subprocess.TimeoutExpired:
        class R: returncode = 124; stdout = ""; stderr = ""; _stdout = ""
        return R()
    r._stdout = r.stdout.strip() if r.stdout else ""
    return r

print("=" * 60)
print("GHCR Push")
print("=" * 60)

# 1) Drive arsivleri tara
image_dir = Path(DRIVE_IMAGE_DIR)
if not image_dir.exists():
    print(f"HATA: {DRIVE_IMAGE_DIR} bulunamadi!")
    sys.exit(1)

archives = sorted(image_dir.glob("*.tar.gz"))
print(f"\nArsiv: {len(archives)} adet")
for a in archives:
    print(f"  {a.name}  ({a.stat().st_size/(1024*1024):.0f} MB)")
if not archives:
    print("Arsiv yok, once build adimlari calistirilmail.")
    sys.exit(1)

# 2) Podman
if run_cap("which podman").returncode != 0:
    print("\nPodman kuruluyor...")
    run_cap("apt-get update -qq")
    run_cap("apt-get install -y -qq podman")
    print("Podman kuruldu.")
else:
    print(f"\nPodman: {run_cap('podman --version')._stdout}")

# 3) GHCR auth
gh_token = None
for key in ["GITHUB_PAT", "GITHUB_TOKEN", "GH_TOKEN"]:
    try:
        val = userdata.get(key)
        if val: gh_token = val; print(f"Token: {key}"); break
    except: pass

if not gh_token:
    print("HATA: GitHub PAT bulunamadi! Colab Secrets > GITHUB_PAT ekleyin.")
    sys.exit(1)

gh_token_val = gh_token.strip()
_auth_b64 = base64.b64encode(f"arda-avci:{gh_token_val}".encode()).decode()
_auth_dir = os.path.expanduser("~/.config/containers")
os.makedirs(_auth_dir, exist_ok=True)
with open(os.path.join(_auth_dir, "auth.json"), "w") as _f:
    json.dump({"auths": {"ghcr.io": {"auth": _auth_b64}}}, _f)

# Token test
_r = requests.get("https://api.github.com/user", headers={"Authorization": f"Bearer {gh_token_val}", "Accept": "application/vnd.github.v3+json"})
if _r.status_code == 200:
    print(f"[OK] GitHub: {_r.json()['login']}")
else:
    print(f"[FAIL] Token gecersiz HTTP {_r.status_code}"); sys.exit(1)

# 4) Skopeo + Crane
if not shutil.which("skopeo"):
    print("Skopeo kuruluyor...")
    run_cap("apt-get install -y -qq skopeo")
if not shutil.which("crane"):
    print("Crane kuruluyor...")
    run_cap("wget -q https://github.com/google/go-containerregistry/releases/latest/download/go-containerregistry_Linux_x86_64.tar.gz -O /tmp/crane.tar.gz && tar -xzf /tmp/crane.tar.gz -C /usr/local/bin/ crane && chmod +x /usr/local/bin/crane", timeout=120)

# 5) Push loop
total = len(archives)
success = 0
failed = []

for idx, archive_path in enumerate(archives, 1):
    model_name = archive_path.stem.replace(".tar", "")
    ghcr_image = f"{GHCR_NAMESPACE}/{IMAGE_PREFIX}{model_name}:latest"
    print(f"\n[{idx}/{total}] {model_name} ({archive_path.stat().st_size/(1024*1024):.0f} MB)")
    print(f"  -> {ghcr_image}")
    sys.stdout.flush()
    start = time.time()
    temp_tar = TEMP_DIR / f"{model_name}.tar"

    # Decompress
    rc, _ = stream_cmd(f"gunzip -c '{archive_path}' > '{temp_tar}'", timeout=7200, step="gunzip")
    if rc != 0 or not temp_tar.exists():
        print(f"  [FAIL] Decompress"); failed.append(model_name); continue

    # Podman load + push
    rc, out = stream_cmd(f"podman load -i '{temp_tar}'", timeout=7200, step="podman load")
    if rc == 0:
        loaded_name = None
        for line in out.split("\n"):
            if "Loaded image" in line:
                parts = line.split(":", 1)
                loaded_name = parts[-1].strip()
                if "sha256:" in loaded_name: loaded_name = None
                break
        if not loaded_name: loaded_name = f"localhost/{IMAGE_PREFIX}{model_name}:local"
        pushed = False
        for attempt in range(3):
            rc2, out2 = stream_cmd(f"podman tag {loaded_name} {ghcr_image} && podman push {ghcr_image}", timeout=1800, step=f"push deneme {attempt+1}/3")
            if rc2 == 0: pushed = True; break
            print(f"    [WARN] {out2[-200:]}")
            if attempt < 2: time.sleep(15)
        run_cap(f"podman rmi {loaded_name} {ghcr_image} 2>/dev/null")
        if pushed:
            print(f"  [OK] {(time.time()-start):.0f}s")
            temp_tar.unlink(missing_ok=True); success += 1; continue

    # Fallback: Skopeo
    if shutil.which("skopeo"):
        rc, _ = stream_cmd(f"skopeo copy --dest-creds=arda-avci:{gh_token_val} docker-archive:'{temp_tar}' docker://{ghcr_image}", timeout=1800, step="skopeo")
        if rc == 0: temp_tar.unlink(missing_ok=True); success += 1; print(f"  [OK] Skopeo {(time.time()-start):.0f}s"); continue

    # Fallback: Crane
    if shutil.which("crane"):
        rc, _ = stream_cmd(f"crane push '{temp_tar}' {ghcr_image}", timeout=1800, step="crane")
        if rc == 0: temp_tar.unlink(missing_ok=True); success += 1; print(f"  [OK] Crane {(time.time()-start):.0f}s"); continue

    print(f"  [FAIL] Tum yontemler basarisiz!")
    failed.append(model_name)
    temp_tar.unlink(missing_ok=True)

print(f"\n=== GHCR: {success}/{total} basarili ===")
if failed: print(f"Basarisiz: {failed}")

## Hucre 8: Ozet ve Sonraki Adimlar

Build tamamlandiginda:
1. GHCR'da imajlar: `ghcr.io/arda-avci/ai-publisher-{model}:latest`
2. Drive'da tgz yedek: `/content/drive/MyDrive/Colab Notebooks/docker/images/`
3. Yerel makineye indir: `colab_docker/pull_images.sh` (Drive'dan kopyala)
4. Docker compose ile baslat: `docker compose -f colab_docker/docker-compose.yml up -d`
5. Servisleri dogrula: `curl http://localhost:5001/health`

### Gerekli Portlar (localhost:5001-5016)
- 5001 cogvideox - Video (I2V)
- 5002 xtts - TTS (Turkce)
- 5003 audioldm2 - SFX
- 5004 wav2lip - Lip-Sync
- 5005 musetalk - Lip-Sync (alternatif)
- 5006 whisper - STT
- 5007 stablediffusion - Gorsel
- 5008 wan - Video (T2V)
- 5009 ltx - Video (I2V)
- 5010 hunyuan - Video
- 5011 kokorotts - TTS (hizli)
- 5012 svd - Video (I2V)
- 5013 animatediff - Video
- 5014 wan25 - Video (genis)
- 5015 f5tts - TTS
- 5016 lora-trainer - LoRA